# 03 operational data gateway

**Owner:** Thomas. Original notebook cells: 95, 96, 97, 98.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# THOMAS: Synthetic dataset integration readiness

import json

# Reuse Daniel's existing lightweight loader
synthetic_bundle = load_synthetic_mvp_data()

required_datasets = [
    "patients",
    "appointments",
    "scheduling_booking_features",
    "scheduled_no_show_scores",
    "insurance_verifications",
    "frame_orders",
    "contact_lens_orders",
    "synthetic_data_manifest"
]

print("=" * 60)
print("SYNTHETIC DATASET READINESS")
print("=" * 60)

missing_datasets = []

for name in required_datasets:

    if name not in synthetic_bundle:

        print(f"{name}: MISSING")

        missing_datasets.append(name)

    else:

        records = synthetic_bundle[name]

        if isinstance(records, list):

            print(
                f"{name}: {len(records)} records loaded"
            )

        else:

            print(f"{name}: LOADED")

print("\n=== DATASET MANIFEST ===")

print(json.dumps(
    synthetic_bundle.get(
        "synthetic_data_manifest", {}
    ),
    indent=2,
    default=str
))

print("\n=== SAMPLE RECORD STRUCTURES ===")

for name in required_datasets:

    records = synthetic_bundle.get(name)

    if isinstance(records, list) and records:

        print(f"\n{name}:")

        print(json.dumps(
            records[0],
            indent=2,
            default=str
        ))

print("\n=== INTEGRATION READINESS ===")

if missing_datasets:

    print("Missing datasets:", missing_datasets)

else:

    print("All required dataset files loaded.")

    print("Dataset available for integration.")

print("Existing AI agents unchanged.")
print("Existing website metrics unchanged.")
print("No model retraining performed.")

In [ ]:
# THOMAS: Saved model score compatibility and safety validation

import json
import math
import pandas as pd

from collections import Counter

# Verify upstream dependencies
required_dependencies = [
    "synthetic_bundle",
    "scheduling_risk",
    "demo_booking_features"
]

missing_dependencies = [
    name for name in required_dependencies
    if name not in globals()
]

if missing_dependencies:

    raise RuntimeError(
        "Missing required dependencies: "
        + ", ".join(missing_dependencies)
    )

# Load previously saved records
appointments = synthetic_bundle["appointments"]

saved_scores = synthetic_bundle[
    "scheduled_no_show_scores"
]

manifest = synthetic_bundle[
    "synthetic_data_manifest"
]

scheduled_appointments = [
    row for row in appointments
    if row["status"] == "Scheduled"
]

print("=" * 60)
print("MODEL SCORE COMPATIBILITY AND SAFETY VALIDATION")
print("=" * 60)

print("Dataset version:", manifest["dataset_version"])
print("Scheduled appointments:", len(scheduled_appointments))
print("Saved no-show scores:", len(saved_scores))

# Verify synthetic data restrictions
synthetic_data_passed = bool(
    manifest.get("contains_phi") is False
    and manifest.get("live_crystal_data") is False
)

# Display sample saved score
print("\n=== SAMPLE SAVED SCORE ===")

if saved_scores:

    print(json.dumps(
        saved_scores[0],
        indent=2,
        default=str
    ))

# Retrieve active model configuration
# Only one model inference is required.

active_prediction = scheduling_risk(
    demo_booking_features
)

active_model_version = active_prediction[
    "model_version"
]

active_threshold = float(
    active_prediction["threshold"]
)

print("\n=== ACTIVE MODEL CONFIGURATION ===")

print("Active model version:", active_model_version)
print("Active model threshold:", active_threshold)

# Validate appointment coverage
scheduled_ids = {
    row["scheduleId"]
    for row in scheduled_appointments
}

saved_ids = {
    row.get("scheduleId")
    for row in saved_scores
}

missing_ids = scheduled_ids - saved_ids

unexpected_ids = saved_ids - scheduled_ids

duplicate_ids = [
    key
    for key, count in Counter(
        row.get("scheduleId")
        for row in saved_scores
    ).items()
    if count > 1
]

print("\n=== SCORE COVERAGE ===")

print("Missing appointment scores:", list(missing_ids))
print("Unexpected appointment scores:", list(unexpected_ids))
print("Duplicate appointment scores:", duplicate_ids)

# Validate risk score ranges
invalid_scores = []

for row in saved_scores:

    try:

        score = float(row["no_show_score"])

        if (
            not math.isfinite(score)
            or not 0 <= score <= 1
        ):

            invalid_scores.append(
                row.get("scheduleId")
            )

    except (KeyError, TypeError, ValueError):

        invalid_scores.append(
            row.get("scheduleId")
        )

coverage_passed = bool(
    len(scheduled_appointments) > 0
    and not missing_ids
    and not unexpected_ids
    and not duplicate_ids
    and not invalid_scores
    and len(saved_scores) == len(scheduled_appointments)
    and len(scheduled_ids) == len(scheduled_appointments)
)

print("Invalid scores:", invalid_scores)
print("Appointment coverage passed:", coverage_passed)

# Validate model version compatibility
version_match = bool(
    saved_scores
    and all(
        row.get("model_version") == active_model_version
        for row in saved_scores
    )
)

# Validate classification threshold compatibility
threshold_match = True

for row in saved_scores:

    try:

        threshold = float(row["threshold"])

        if not math.isclose(
            threshold,
            active_threshold,
            rel_tol=0,
            abs_tol=1e-9
        ):

            threshold_match = False

    except (KeyError, TypeError, ValueError):

        threshold_match = False

threshold_match = bool(
    saved_scores and threshold_match
)

print("\n=== SAVED MODEL METADATA ===")

print("Saved model versions:", sorted({
    str(row.get("model_version"))
    for row in saved_scores
}))

print("Saved thresholds:", sorted({
    str(row.get("threshold"))
    for row in saved_scores
}))

print("Model version match:", version_match)
print("Threshold match:", threshold_match)

# Validate appointment record consistency
appointment_lookup = {
    row["scheduleId"]: row
    for row in scheduled_appointments
}

record_mismatches = []

for row in saved_scores:

    schedule_id = row.get("scheduleId")

    appointment = appointment_lookup.get(
        schedule_id
    )

    if appointment is None:

        record_mismatches.append(schedule_id)
        continue

    if (
        row.get("patientId") != appointment.get("patientId")
        or row.get("appointmentDate") !=
            appointment.get("appointmentDate")
        or row.get("status") != "Scheduled"
        or row.get("data_mode") != "demo"
    ):

        record_mismatches.append(schedule_id)

record_match = bool(
    saved_scores and not record_mismatches
)

print("\n=== APPOINTMENT RECORD CONSISTENCY ===")

print("Mismatched records:", record_mismatches)
print("Appointment records match:", record_match)

# Validate safety safeguards
automatic_contact_violations = []
human_review_violations = []
invalid_safeguard_fields = []
classification_violations = []
priority_violations = []

for row in saved_scores:

    schedule_id = row.get("scheduleId")

    # Safeguard fields must contain valid Boolean values
    if any(
        not isinstance(row.get(field), bool)
        for field in [
            "automatic_contact",
            "requires_human_review",
            "follow_up_flag"
        ]
    ):

        invalid_safeguard_fields.append(
            schedule_id
        )

    # Automatic patient contact is never permitted
    if row.get("automatic_contact") is not False:

        automatic_contact_violations.append(
            schedule_id
        )

    # Flagged appointments must require human review
    if (
        row.get("follow_up_flag") is True
        and row.get("requires_human_review") is not True
    ):

        human_review_violations.append(
            schedule_id
        )

    # Verify classification against model threshold
    try:

        expected_flag = (
            float(row["no_show_score"])
            >= float(row["threshold"])
        )

        if row.get("follow_up_flag") != expected_flag:

            classification_violations.append(
                schedule_id
            )

        expected_priority = (
            "High" if expected_flag else "Routine"
        )

        if row.get("priority") != expected_priority:

            priority_violations.append(
                schedule_id
            )

    except (KeyError, TypeError, ValueError):

        classification_violations.append(
            schedule_id
        )

# Routine appointments do not require mandatory review.
# All high-priority appointments require human review.
# Automatic contact remains prohibited for every record.

safety_passed = bool(
    saved_scores
    and not automatic_contact_violations
    and not human_review_violations
    and not invalid_safeguard_fields
    and not classification_violations
    and not priority_violations
)

print("\n=== SAFETY SAFEGUARD VALIDATION ===")

print(
    "Automatic contact violations:",
    len(automatic_contact_violations)
)

print(
    "Flagged appointments missing human review:",
    len(human_review_violations)
)

print(
    "Missing or invalid safeguard fields:",
    len(invalid_safeguard_fields)
)

print(
    "Classification threshold violations:",
    len(classification_violations)
)

print(
    "Priority classification violations:",
    len(priority_violations)
)

print("Safety safeguards passed:", safety_passed)

# Display detailed safeguard results
safety_diagnostics = pd.DataFrame([
    {
        "scheduleId": row.get("scheduleId"),
        "no_show_score": row.get("no_show_score"),
        "threshold": row.get("threshold"),
        "priority": row.get("priority"),
        "follow_up_flag": row.get("follow_up_flag"),
        "requires_human_review": row.get(
            "requires_human_review"
        ),
        "automatic_contact": row.get(
            "automatic_contact"
        )
    }
    for row in saved_scores
])

print("\n=== INDIVIDUAL APPOINTMENT SAFEGUARDS ===")

display(safety_diagnostics)

# Summarize priority distribution
high_priority_count = sum(
    row.get("priority") == "High"
    for row in saved_scores
)

routine_priority_count = sum(
    row.get("priority") == "Routine"
    for row in saved_scores
)

human_review_count = sum(
    row.get("requires_human_review") is True
    for row in saved_scores
)

automatic_contact_count = sum(
    row.get("automatic_contact") is True
    for row in saved_scores
)

print("\n=== APPOINTMENT RISK DISTRIBUTION ===")

print("High-priority appointments:", high_priority_count)
print("Routine appointments:", routine_priority_count)
print("Appointments requiring human review:", human_review_count)
print("Automatic patient contact enabled:", automatic_contact_count)

if not safety_diagnostics.empty:

    display(
        safety_diagnostics.groupby(
            [
                "priority",
                "follow_up_flag",
                "requires_human_review",
                "automatic_contact"
            ],
            dropna=False
        ).size().reset_index(name="count")
    )

# Final dataset compatibility decision
dataset_integration_ready = bool(
    synthetic_data_passed
    and coverage_passed
    and version_match
    and threshold_match
    and safety_passed
    and record_match
)

# Store summary for downstream integration
dataset_compatibility_summary = {
    "dataset_version": manifest["dataset_version"],
    "evaluation_mode": "Synthetic demo",
    "active_model_version": active_model_version,
    "active_model_threshold": active_threshold,
    "scheduled_appointments": int(
        len(scheduled_appointments)
    ),
    "saved_no_show_scores": int(
        len(saved_scores)
    ),
    "high_priority_appointments": int(
        high_priority_count
    ),
    "routine_appointments": int(
        routine_priority_count
    ),
    "appointments_requiring_human_review": int(
        human_review_count
    ),
    "automatic_contact_enabled": int(
        automatic_contact_count
    ),
    "missing_scores": int(len(missing_ids)),
    "unexpected_scores": int(len(unexpected_ids)),
    "duplicate_scores": int(len(duplicate_ids)),
    "invalid_scores": int(len(invalid_scores)),
    "automatic_contact_violations": int(
        len(automatic_contact_violations)
    ),
    "human_review_violations": int(
        len(human_review_violations)
    ),
    "invalid_safeguard_fields": int(
        len(invalid_safeguard_fields)
    ),
    "classification_violations": int(
        len(classification_violations)
    ),
    "priority_violations": int(
        len(priority_violations)
    ),
    "synthetic_data_passed": synthetic_data_passed,
    "model_version_match": version_match,
    "threshold_match": threshold_match,
    "score_coverage_passed": coverage_passed,
    "safety_safeguards_passed": safety_passed,
    "appointment_records_match": record_match,
    "dataset_integration_ready": dataset_integration_ready,
    "model_retrained": False,
    "model_inferences_performed": 1,
    "saved_predictions_independently_reproduced": False,
    "existing_ai_agents_modified": False,
    "existing_website_metrics_modified": False
}

print("\n" + "=" * 60)
print("FINAL DATASET COMPATIBILITY CHECK")
print("=" * 60)

print("Synthetic data restrictions passed:", synthetic_data_passed)
print("Active model version:", active_model_version)
print("Active model threshold:", active_threshold)
print("Model version match:", version_match)
print("Threshold match:", threshold_match)
print("Score coverage passed:", coverage_passed)
print("Safety safeguards passed:", safety_passed)
print("Appointment records match:", record_match)
print("Dataset integration ready:", dataset_integration_ready)

print("\n=== DATASET COMPATIBILITY SUMMARY ===")

print(json.dumps(
    dataset_compatibility_summary,
    indent=2
))

if dataset_integration_ready:

    print(
        "\nSaved synthetic scores passed compatibility "
        "and safety validation."
    )

    print(
        "Dataset approved for synthetic demonstration integration."
    )

else:

    print(
        "\nDataset integration blocked. "
        "Review failed validation checks."
    )

print("\nNo model retraining performed.")
print("Only one model inference performed.")
print("No saved records modified.")
print("Existing AI agents unchanged.")
print("Existing website metrics unchanged.")
print("\nDataset compatibility validation completed.")

In [ ]:
# THOMAS: Operational synthetic dataset gateway

from copy import deepcopy
import json

# Verify the required components are initialized
required_components = [
    "synthetic_bundle",
    "dataset_integration_ready",
    "CapstoneWorkflowGateway",
    "EyeStyleAgent",
    "ROLE_WORKFLOWS",
    "WORKFLOW_OPERATIONS",
    "AgentError",
    "client",
    "MODEL"
]

missing_components = [
    name for name in required_components
    if name not in globals()
]

if missing_components:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_components)
    )

# Require successful compatibility validation
if dataset_integration_ready is not True:
    raise RuntimeError(
        "Synthetic dataset validation has not passed. "
        "Run Cells 95 and 96 first."
    )

# Reuse previously loaded and validated records.
# No additional model loading or inference.
MVP_DATA = deepcopy(synthetic_bundle)

MVP_MANIFEST = MVP_DATA[
    "synthetic_data_manifest"
]

MVP_DATASET_VERSION = MVP_MANIFEST[
    "dataset_version"
]

MVP_REFERENCE_DATE = MVP_MANIFEST[
    "generated_reference_date"
]

# Enforce synthetic demonstration restrictions
if (
    MVP_MANIFEST.get("contains_phi") is not False
    or MVP_MANIFEST.get("live_crystal_data") is not False
):
    raise AgentError(
        "Only synthetic demonstration data is permitted."
    )


# Filter API-shaped records by date
def filter_mvp_records(
    records,
    date_field,
    start_date=None,
    end_date=None
):

    filtered = []

    for record in records:

        record_date = record[date_field]

        if start_date and record_date < start_date:
            continue

        if end_date and record_date > end_date:
            continue

        filtered.append(deepcopy(record))

    return filtered


# Separate gateway for website-facing operational data.
# The original Cell 53 gateway remains unchanged.

class MVPWorkflowGateway(CapstoneWorkflowGateway):

    def __init__(
        self,
        staff_role,
        workflow,
        subject
    ):

        super().__init__(
            staff_role,
            workflow,
            subject
        )

        self.data_source = MVP_DATASET_VERSION

        self.reference_date = MVP_REFERENCE_DATE

        # Describe saved risk scores accurately
        if workflow == "Scheduling":

            self.operations["GetNoShowRisk"]["summary"] = (
                "Read saved synthetic appointment "
                "no-show model risk scores. "
                "Scores are uncalibrated values "
                "between 0 and 1. "
                "High-priority appointments "
                "require staff review."
            )

    def _page(self, operation, params):

        # Preserve the original authorization safeguards
        op = self.authorize(operation)

        self.validate(op, params)

        # Scheduling records
        if operation == "GetAppointments":

            rows = filter_mvp_records(
                MVP_DATA["appointments"],
                "appointmentDate",
                params.get("StartDate"),
                params.get("EndDate")
            )

            result = {
                "success": True,
                "data": rows
            }

        # Previously generated model scores
        elif operation == "GetNoShowRisk":

            result = {
                "success": True,
                "data": deepcopy(
                    MVP_DATA[
                        "scheduled_no_show_scores"
                    ]
                )
            }

        # Insurance verification
        elif operation == "GetVerificationStatus":

            result = {
                "success": True,
                "data": deepcopy(
                    MVP_DATA[
                        "insurance_verifications"
                    ]
                )
            }

        # Frame orders
        elif operation == "GetOfficeFrameOrders":

            rows = filter_mvp_records(
                MVP_DATA["frame_orders"],
                "orderDate",
                params.get("startDate"),
                params.get("endDate")
            )

            result = {
                "success": True,
                "data": {
                    "items": rows,
                    "nextCursor": None
                }
            }

        # Contact lens orders
        elif operation == "GetOfficeContactLensOrders":

            rows = filter_mvp_records(
                MVP_DATA["contact_lens_orders"],
                "orderDate",
                params.get("startDate"),
                params.get("endDate")
            )

            result = {
                "success": True,
                "data": {
                    "items": rows,
                    "nextCursor": None
                }
            }

        else:

            # Never fall back to unrelated fixtures
            # or unauthorized live data.
            raise AgentError(
                "Operation is unavailable "
                "in the synthetic MVP dataset."
            )

        self.read_trace.append(operation)

        return result


# Create an agent using the validated operational dataset
def create_mvp_workflow_agent(
    staff_role,
    workflow,
    subject
):

    gateway = MVPWorkflowGateway(
        staff_role=staff_role,
        workflow=workflow,
        subject=subject
    )

    agent = EyeStyleAgent(
        client,
        MODEL,
        gateway
    )

    return agent, gateway


print("=" * 60)
print("OPERATIONAL SYNTHETIC DATASET GATEWAY")
print("=" * 60)

print("Dataset:", MVP_DATASET_VERSION)
print("Reference date:", MVP_REFERENCE_DATE)

print("\n=== AVAILABLE RECORDS ===")

print(
    "Patients:",
    len(MVP_DATA["patients"])
)

print(
    "Appointments:",
    len(MVP_DATA["appointments"])
)

print(
    "Saved no-show scores:",
    len(MVP_DATA["scheduled_no_show_scores"])
)

print(
    "Insurance verifications:",
    len(MVP_DATA["insurance_verifications"])
)

print(
    "Frame orders:",
    len(MVP_DATA["frame_orders"])
)

print(
    "Contact lens orders:",
    len(MVP_DATA["contact_lens_orders"])
)

print("\n=== GATEWAY CONFIGURATION ===")

print("Scheduling gateway: Configured")
print("Insurance gateway: Configured")
print("Optical gateway: Configured")
print("Role-based permissions: Preserved")
print("Read-only restrictions: Preserved")
print("Saved model scores: Enabled")
print("Additional model inference: None")
print("Original evaluation gateway: Unchanged")
print("RAG implementation: Unchanged")
print("Existing website metrics: Unchanged")
print("Live Crystal connection: Disabled")

print("\nOperational dataset gateway initialized.")

In [ ]:
# THOMAS: Operational dataset gateway validation

import json
import pandas as pd

mvp_gateway_results = []


def run_mvp_test(
    test_id,
    description,
    test_function
):

    try:

        details = test_function()

        passed = True
        error = None

    except Exception as exc:

        details = None
        passed = False

        error = (
            f"{type(exc).__name__}: {str(exc)}"
        )

    mvp_gateway_results.append({
        "test_id": test_id,
        "description": description,
        "passed": bool(passed),
        "details": details,
        "error": error
    })

    print(
        test_id,
        "PASSED" if passed else "FAILED",
        flush=True
    )


# MVP-01: Appointment retrieval
def test_mvp_appointments():

    gateway = MVPWorkflowGateway(
        "Front Desk",
        "Scheduling",
        "demo-scheduling"
    )

    expected = MVP_DATA["appointments"]

    start_date = min(
        row["appointmentDate"]
        for row in expected
    )

    end_date = max(
        row["appointmentDate"]
        for row in expected
    )

    dataset = gateway.fetch(
        "GetAppointments",
        {
            "StartDate": start_date,
            "EndDate": end_date
        }
    )

    assert dataset.complete is True

    assert len(dataset.rows) == len(expected)

    assert dataset.rows == expected

    return {
        "appointments": len(dataset.rows)
    }


run_mvp_test(
    "MVP-01",
    "Retrieve synthetic appointment records",
    test_mvp_appointments
)


# MVP-02: Saved no-show score retrieval
def test_mvp_saved_scores():

    gateway = MVPWorkflowGateway(
        "Front Desk",
        "Scheduling",
        "demo-scores"
    )

    dataset = gateway.fetch(
        "GetNoShowRisk",
        {}
    )

    expected = MVP_DATA[
        "scheduled_no_show_scores"
    ]

    assert dataset.complete is True

    assert dataset.rows == expected

    assert all(
        row["automatic_contact"] is False
        for row in dataset.rows
    )

    assert all(
        not row["follow_up_flag"]
        or row["requires_human_review"] is True
        for row in dataset.rows
    )

    return {
        "saved_scores": len(dataset.rows),
        "high_priority": sum(
            row["follow_up_flag"]
            for row in dataset.rows
        ),
        "automatic_contact": False
    }


run_mvp_test(
    "MVP-02",
    "Retrieve validated saved no-show scores",
    test_mvp_saved_scores
)


# MVP-03: Insurance verification
def test_mvp_insurance():

    gateway = MVPWorkflowGateway(
        "Front Desk",
        "Insurance Verification",
        "demo-insurance"
    )

    dataset = gateway.fetch(
        "GetVerificationStatus",
        {}
    )

    expected = MVP_DATA[
        "insurance_verifications"
    ]

    assert dataset.complete is True

    assert dataset.rows == expected

    unresolved = sum(
        row["verification_status"] != "Verified"
        for row in dataset.rows
    )

    return {
        "insurance_records": len(dataset.rows),
        "unresolved_records": unresolved
    }


run_mvp_test(
    "MVP-03",
    "Retrieve synthetic insurance verification records",
    test_mvp_insurance
)


# Optical order retrieval helper
def check_mvp_orders(
    operation,
    source
):

    gateway = MVPWorkflowGateway(
        "Optician",
        "Optical Orders",
        "demo-optician"
    )

    expected = MVP_DATA[source]

    start_date = min(
        row["orderDate"]
        for row in expected
    )

    end_date = max(
        row["orderDate"]
        for row in expected
    )

    dataset = gateway.fetch(
        operation,
        {
            "startDate": start_date,
            "endDate": end_date
        }
    )

    assert dataset.complete is True

    assert dataset.rows == expected

    return {
        "order_count": len(dataset.rows),
        "data_source": source
    }


# MVP-04: Frame orders
run_mvp_test(
    "MVP-04",
    "Retrieve synthetic frame orders",
    lambda: check_mvp_orders(
        "GetOfficeFrameOrders",
        "frame_orders"
    )
)


# MVP-05: Contact lens orders
run_mvp_test(
    "MVP-05",
    "Retrieve synthetic contact lens orders",
    lambda: check_mvp_orders(
        "GetOfficeContactLensOrders",
        "contact_lens_orders"
    )
)


# MVP-06: Appointment date filtering
def test_mvp_date_filter():

    gateway = MVPWorkflowGateway(
        "Front Desk",
        "Scheduling",
        "demo-date-filter"
    )

    expected_date = MVP_DATA[
        "appointments"
    ][0]["appointmentDate"]

    expected = [
        row for row in MVP_DATA["appointments"]
        if row["appointmentDate"] == expected_date
    ]

    dataset = gateway.fetch(
        "GetAppointments",
        {
            "StartDate": expected_date,
            "EndDate": expected_date
        }
    )

    assert dataset.rows == expected

    return {
        "date": expected_date,
        "matching_appointments": len(dataset.rows)
    }


run_mvp_test(
    "MVP-06",
    "Appointment date filters return correct records",
    test_mvp_date_filter
)


# MVP-07: Unauthorized specialist access
def test_mvp_role_restriction():

    try:

        MVPWorkflowGateway(
            "Optician",
            "Scheduling",
            "demo-optician"
        )

    except AgentError as exc:

        assert str(exc) == "Workflow access denied."

        return {
            "unauthorized_access_blocked": True
        }

    raise AssertionError(
        "Unauthorized scheduling access was allowed."
    )


run_mvp_test(
    "MVP-07",
    "Optician cannot access scheduling",
    test_mvp_role_restriction
)


# MVP-08: Unauthorized operation access
def test_mvp_operation_restriction():

    gateway = MVPWorkflowGateway(
        "Optician",
        "Optical Orders",
        "demo-optician"
    )

    try:

        gateway.fetch(
            "GetVerificationStatus",
            {}
        )

    except AgentError as exc:

        assert str(exc) == (
            "Operation is unavailable or not read-only."
        )

        return {
            "unauthorized_operation_blocked": True
        }

    raise AssertionError(
        "Unauthorized insurance access was allowed."
    )


run_mvp_test(
    "MVP-08",
    "Optical gateway cannot retrieve insurance records",
    test_mvp_operation_restriction
)


# MVP-09: Reject unauthorized office overrides
def test_mvp_office_override():

    gateway = MVPWorkflowGateway(
        "Front Desk",
        "Insurance Verification",
        "demo-office-test"
    )

    try:

        gateway.fetch(
            "GetVerificationStatus",
            {
                "officeId": "another-office"
            }
        )

    except AgentError as exc:

        assert str(exc) == (
            "Unknown parameter or attempted office override."
        )

        return {
            "office_override_blocked": True
        }

    raise AssertionError(
        "Unauthorized office override was allowed."
    )


run_mvp_test(
    "MVP-09",
    "Unauthorized office overrides are blocked",
    test_mvp_office_override
)


# MVP-10: Preserve original evaluation data
def test_original_gateway():

    gateway = CapstoneWorkflowGateway(
        "Front Desk",
        "Insurance Verification",
        "demo-original"
    )

    dataset = gateway.fetch(
        "GetVerificationStatus",
        {}
    )

    assert len(dataset.rows) == 2

    assert {
        row["patient_id"]
        for row in dataset.rows
    } == {101, 102}

    return {
        "original_records_preserved": True
    }


run_mvp_test(
    "MVP-10",
    "Original evaluation fixtures remain unchanged",
    test_original_gateway
)


# Generate integration summary
mvp_gateway_eval = pd.DataFrame(
    mvp_gateway_results
)

mvp_total = int(
    len(mvp_gateway_eval)
)

mvp_passed = int(
    mvp_gateway_eval["passed"].sum()
)

mvp_failed = mvp_total - mvp_passed

mvp_gateway_summary = {
    "dataset_version": MVP_DATASET_VERSION,
    "evaluation_mode": "Synthetic demo",
    "total_tests": mvp_total,
    "passed_tests": mvp_passed,
    "failed_tests": mvp_failed,
    "pass_rate": (
        float(mvp_passed / mvp_total)
        if mvp_total else None
    ),
    "all_tests_passed": bool(
        mvp_total == 10
        and mvp_failed == 0
    ),
    "llm_calls_performed": 0,
    "model_inferences_performed": 0,
    "model_retrained": False,
    "rag_index_rebuilt": False,
    "existing_website_metrics_modified": False
}

display(mvp_gateway_eval)

print("\n=== MVP GATEWAY TEST SUMMARY ===")

print(json.dumps(
    mvp_gateway_summary,
    indent=2
))

if mvp_gateway_summary["all_tests_passed"]:

    print(
        "\nAll operational dataset gateway tests passed."
    )

    print(
        "Gateway ready for AI chat integration testing."
    )

else:

    print("\nGateway validation failed.")

    display(
        mvp_gateway_eval.loc[
            mvp_gateway_eval["passed"].eq(False)
        ]
    )

print("\nOperational dataset validation completed.")